# Perceiver CPI — does a pretrained protein encoder help?

We measured that our models **rank proteins no better than chance**: keep the drug the same, change the protein, and the answer barely moves. The app depends on that working.

The suspected cause: the paper builds its protein encoder from scratch and learns what amino acids mean from only **243 proteins**. This tests replacing it with **ESM-2**, a protein model pretrained on millions of sequences.

| Group | Protein encoder | Runs |
|---|---|---|
| `baseline` | the paper's original | 2 seeds × 400 epochs |
| `esm` | ESM-2 | 2 seeds × 400 epochs |

**Run one group per session**, about 5–6 hours each. Both fit inside Kaggle's 12-hour limit with room to spare.

### Before you run
1. **Add Input** (right panel) → **Datasets** → add your `perceiver cpi data` dataset.
2. **Settings → Accelerator → GPU T4 x2**.
3. Set `GROUP` in the next cell: `'baseline'` the first time, `'esm'` the second.
4. **Save Version → Save & Run All (Commit)**, then close the browser.

### Second session
When you come back for the other group, also **Add Input → Your Work → Notebooks** and attach this notebook's previous output. Finished runs are then skipped and nothing is repeated.

In [ ]:
# 1. Which half to run, and find the data
GROUP = 'baseline'      # <-- 'baseline' first, then 'esm' in a second session

import os, shutil, subprocess, sys
from pathlib import Path

INPUT = Path(os.environ.get('KAGGLE_INPUT', '/kaggle/input'))
WORK = Path(os.environ.get('KAGGLE_WORKING', '/kaggle/working'))

def find_source():
    for depth in range(1, 6):
        found = sorted(INPUT.glob('/'.join(['*'] * depth) + '/run_experiment.py'))
        if found:
            return found[0].parent
    return None

SOURCE = find_source()
if SOURCE is None:
    listing = '\n'.join(f'  {p.relative_to(INPUT)}' for p in sorted(INPUT.rglob('*'))[:40]) or '  (nothing)'
    raise FileNotFoundError(
        'Could not find run_experiment.py in the attached data.\n'
        'Use "Add Input" (right panel) -> Datasets -> add the dataset you uploaded, then Run All again.\n'
        f'What is attached right now:\n{listing}')
print('Found the data at:', SOURCE)

CODE = WORK / 'perceiver'
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
for item in SOURCE.iterdir():
    (shutil.copytree if item.is_dir() else shutil.copy2)(item, CODE / item.name)
print((CODE / 'BUNDLE_INFO.txt').read_text())

RUNS = WORK / 'runs'
RUNS.mkdir(exist_ok=True)
os.environ['PERCEIVER_RUNS'] = str(RUNS)

# carry over a previous session's progress, if its output was attached as an input
for previous in sorted(INPUT.glob('*/runs')):
    for folder in previous.iterdir():
        if folder.is_dir() and not (RUNS / folder.name).exists():
            shutil.copytree(folder, RUNS / folder.name)
            print('Carried over:', folder.name)
print('\nRunning group:', GROUP)

In [ ]:
# 2. Show the GPU and install what the model needs (2-3 minutes)
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                         capture_output=True, text=True).stdout)
else:
    print('No GPU found. Set Settings -> Accelerator -> GPU T4 x2, then Run All again.')
print('CPU cores:', os.cpu_count())

packages = ['rdkit', 'tape-proteins', 'typed-argument-parser', 'tensorboardX', 'lifelines']
if GROUP == 'esm':
    packages.append('fair-esm')
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages],
                        capture_output=True, text=True)
if result.returncode != 0:
    print(result.stdout[-3000:], result.stderr[-3000:])
    raise RuntimeError('Installing packages failed (see above).')

print(subprocess.run([sys.executable, '-c',
                      'import torch; print("PyTorch", torch.__version__, "| GPU:",'
                      ' torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")'],
                     capture_output=True, text=True).stdout)

In [ ]:
# 3. For the ESM group only: work out each protein's features once (a few minutes on the GPU).
#    ESM is frozen, so its output never changes - computing it every epoch would waste the whole run.
if GROUP == 'esm':
    bank = CODE / 'esm_bank_davis.npz'
    if bank.exists():
        print('bank already present:', bank.stat().st_size / 1024**2, 'MB')
    else:
        build = subprocess.run([sys.executable, 'make_esm_bank.py', 'davis'],
                               cwd=CODE, capture_output=True, text=True)
        print(build.stdout[-2500:], build.stderr[-2000:])
        if build.returncode != 0 or not bank.exists():
            raise RuntimeError('Building the ESM features failed (see above).')
else:
    print('baseline group: no ESM features needed.')

In [ ]:
# 4. Quick check: one epoch of the first run in this group (about 2 minutes)
if any(RUNS.iterdir()):
    print('Resuming a previous session, so the check is skipped.')
else:
    check = subprocess.run([sys.executable, 'run_experiment.py', '--group', GROUP, '--epochs', '1'],
                           cwd=CODE, capture_output=True, text=True,
                           env={**os.environ, 'PERCEIVER_RUNS': str(WORK / 'check')})
    print(check.stdout[-2500:], check.stderr[-2500:])
    if check.returncode != 0:
        raise RuntimeError('The check failed (see above).')
    shutil.rmtree(WORK / 'check', ignore_errors=True)
    print('\nCheck passed.')

In [ ]:
# 5. Train this group (about 5-6 hours). A progress bar updates about once a minute.
os.chdir(CODE)
get_ipython().system(f'"{sys.executable}" run_experiment.py --group {GROUP} {os.environ.get("PERCEIVER_TEST_ARGS", "")}')

In [ ]:
# 6. Results so far. The full comparison needs both groups, so after the first session this
#    will only show half of them.
for script in ('evaluate.py', 'protein_sensitivity.py'):
    out = subprocess.run([sys.executable, script], cwd=CODE, capture_output=True, text=True)
    print(f'----- {script} -----')
    print(out.stdout[-3000:] or out.stderr[-1500:])
for name in ('results.md', 'protein_sensitivity.md'):
    if (CODE / name).exists():
        shutil.copy2(CODE / name, WORK / name)

## When it finishes

Everything in `/kaggle/working` is saved as this notebook's output.

**If you have just run `baseline`:** change `GROUP` to `'esm'` in cell 1, attach this notebook's output as an input (Add Input → Your Work → Notebooks), and run again.

**Once both groups are done:** send Claude the output. The quickest way is the command Kaggle shows at the bottom of the Output tab.

The number that matters is **protein ranking** — whether changing the protein changes the answer in the right direction. Our models have been at 0.00, and the ceiling is about +0.41.